# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, their fields and column `@id`s.

Below, we enumerate all record set `@id`s, along with their field `@id`s.

In [ ]:
# List all record sets, their @ids, and fields (by @id)
record_sets = dataset.metadata.get('recordSet', [])
if not record_sets:
    print('No record sets found in the metadata.')
else:
    for rset in record_sets:
        print(f"RecordSet:")
        if isinstance(rset, dict):
            rset_id = rset.get('@id', str(rset))
            print(f"  @id: {rset_id}")
        else:
            rset_id = getattr(rset, '@id', str(rset))
            print(f"  @id: {rset_id}")
            rset = rset.to_json() if hasattr(rset, 'to_json') else rset
        # Fields
        fields = rset.get('field', []) if isinstance(rset, dict) else []
        if fields:
            print("    Fields (by @id):")
            for field in fields:
                fid = field.get('@id', str(field)) if isinstance(field, dict) else field
                print(f"      - {fid}")
        # Columns
        columns = rset.get('column', []) if isinstance(rset, dict) else []
        if columns:
            print("    Columns (by @id):")
            for col in columns:
                cid = col.get('@id', str(col)) if isinstance(col, dict) else col
                print(f"      - {cid}")
        print()

# If no recordSet found, we can attempt to enumerate distributions
if not record_sets:
    distributions = dataset.metadata.get('distribution', [])
    print('Distributions available (@id):')
    for d in distributions:
        print(f"  - {d.get('@id', str(d)) if isinstance(d, dict) else d}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

> **Note:** As the FAIR² Mass Spectrometry dataset may not define any `recordSet` entities in its root schema but only has `distribution` entries (e.g., a tabular data file), we will read the available distribution(s) as record sources.

In [ ]:
# As there are no 'recordSet', extract from available distributions as record sets
dist_keys = []
dist_objs = dataset.metadata.get('distribution', [])
if not isinstance(dist_objs, list):
    dist_objs = [dist_objs]
for d in dist_objs:
    did = d.get('@id', str(d)) if isinstance(d, dict) else d
    dist_keys.append(did)
print('Distributions found as record sets:', dist_keys)

dataframes = {}
for dist_id in dist_keys:
    try:
        records = list(dataset.records(record_set=dist_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            print(f"Loaded DataFrame from distribution @id={dist_id} (shape: {df.shape})")
        else:
            print(f"No records found for distribution @id={dist_id}")
    except Exception as e:
        print(f"Failed to load distribution {dist_id}: {e}")

# For demonstration, display columns for the first available DataFrame
if dataframes:
    main_id = list(dataframes.keys())[0]
    print(f"Fields for DataFrame loaded from: {main_id}")
    print(dataframes[main_id].columns.tolist())
    dataframes[main_id].head()
else:
    print('No tabular data could be loaded from available distribution record sets.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing a numeric column, and grouping by a categorical field.

Below, we select a numeric field for filtering and normalization. Update the `numeric_field` and `group_field` below to those available in the loaded DataFrame columns.

In [ ]:
# Set record_set_id to the @id of the distribution used as record set
record_set_id = main_id if dataframes else None

if record_set_id:
    df = dataframes[record_set_id]

    # List columns for user
    print('Available columns:', df.columns.tolist())

    # Try to choose a numeric field (by heuristic: columns with float/int data)
    numeric_candidates = df.select_dtypes(include=["float", "int"]).columns.tolist()
    print('Numeric field candidates:', numeric_candidates)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Fallback: try to coerce columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_field = col
                break
            except Exception:
                continue
        else:
            print('No suitable numeric field found for EDA.')
            numeric_field = None

    if numeric_field:
        print(f'Using numeric field: {numeric_field}')
        threshold = df[numeric_field].mean()
        print(f'Filtering: {numeric_field} > {threshold}')
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

        # Try to choose a group field (categorical)
        categorical_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
        if len(categorical_candidates) > 0:
            group_field = categorical_candidates[0]
            print(f'Grouping by field: {group_field}')
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped {numeric_field} mean by {group_field}:")
            print(grouped_df.head())
        else:
            print('No categorical field available for groupby demonstration.')
    else:
        print('Skipping EDA: No numeric field found in available columns.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and record_set_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: No numeric field available or no data loaded.')

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect a Croissant-formatted dataset using the `mlcroissant` library
- List available record sets (using their `@id`) and columns
- Extract records as DataFrames, filter and normalize numerics, and group by categorical fields
- Visualize data distributions

You can extend this analysis by using specific `@id`s for fields and deeper domain-specific data wrangling or modeling.
